In [1]:
import torch
print(torch.backends.mps.is_available())
print(torch.backends.mps.is_built())

True
True


In [3]:
from rdkit import Chem
from rdkit.Chem import Descriptors, QED, rdMolDescriptors
from rdkit.Chem import rdFingerprintGenerator
import pandas as pd
import torch
import numpy as np
from multiprocessing import Pool, cpu_count
from chembl_webresource_client.new_client import new_client

device = torch.device("mps")
print(f"Using device: {device}")
print(f"CPUs available: {cpu_count()}")

# ── Step 1: Load candidates ────────────────────────────────────
url        = "https://raw.githubusercontent.com/aspuru-guzik-group/chemical_vae/master/models/zinc_properties/250k_rndm_zinc_drugs_clean_3.csv"
df         = pd.read_csv(url)
candidates = df["smiles"].tolist()[:20000]
print(f"Candidates: {len(candidates)}")

# ── Step 2: Load ChEMBL reference ─────────────────────────────
activity      = new_client.activity
acts          = activity.filter(
                    target_chembl_id="CHEMBL1781",
                    standard_type="IC50"
                ).only("canonical_smiles")
chembl_smiles = [a["canonical_smiles"] for a in acts if a["canonical_smiles"]]
print(f"Reference: {len(chembl_smiles)}")

# ── Step 3: Encode function ────────────────────────────────────
def encode(smi):
    try:
        mol = Chem.MolFromSmiles(smi.strip())
        if mol is None: return None
        gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
        return list(gen.GetFingerprint(mol))
    except:
        return None

# ── Step 4: Encode reference in parallel (CPU) ────────────────
# Step 4 — encode reference
print("Encoding reference...")
with ThreadPoolExecutor() as ex:
    ref_raw = list(ex.map(encode, chembl_smiles))
ref_valid = [fp for fp in ref_raw if fp is not None]
print(f"Valid reference: {len(ref_valid)}")

# Step 5 — encode candidates
print("Encoding candidates...")
with ThreadPoolExecutor() as ex:
    cand_raw = list(ex.map(encode, candidates))

candidate_fps = [
    (smi, fp)
    for smi, fp in zip(candidates, cand_raw)
    if fp is not None
]
print(f"Valid candidates: {len(candidate_fps)}")

# ── Step 6: Score on Mac GPU in batches ───────────────────────
print("Scoring on Mac GPU...")
BATCH   = 512
results = []

smiles_list = [smi for smi, _ in candidate_fps]
fps_list    = [fp  for _,  fp in candidate_fps]

for i in range(0, len(fps_list), BATCH):
    batch_fps  = fps_list[i:i+BATCH]
    batch_smi  = smiles_list[i:i+BATCH]

    # move batch to GPU
    query = torch.tensor(
        batch_fps,
        dtype=torch.float32
    ).to(device)                          # shape: [batch, 2048]

    # vectorised Tanimoto — all queries vs all references at once
    inter  = torch.mm(query, ref_tensor.T)                    # [batch, ref]
    query_sum = query.sum(dim=1, keepdim=True)                # [batch, 1]
    ref_sum   = ref_tensor.sum(dim=1, keepdim=True)           # [ref, 1]
    union     = query_sum + ref_sum.T - inter                 # [batch, ref]
    tanimoto  = (inter / union).max(dim=1).values             # [batch]

    # move back to CPU
    scores = tanimoto.cpu().numpy()

    for smi, score in zip(batch_smi, scores):
        results.append({"smiles": smi, "tanimoto": float(score)})

    print(f"  processed {min(i+BATCH, len(fps_list))}/{len(fps_list)}")

# ── Step 7: Results ────────────────────────────────────────────
df_results = pd.DataFrame(results)
hits       = df_results[df_results["tanimoto"] > 0.40].copy()

print(f"\nScored:     {len(df_results)}")
print(f"Hits >0.40: {len(hits)}")
print(f"Hits >0.60: {(df_results['tanimoto'] > 0.60).sum()}")
print(f"Hits >0.80: {(df_results['tanimoto'] > 0.80).sum()}")
print("\nTanimoto distribution:")
print(df_results["tanimoto"].describe())

Using device: mps
CPUs available: 8
Candidates: 20000
Reference: 735
Encoding reference...


NameError: name 'ThreadPoolExecutor' is not defined